# Prediksi Tinggi Muka Air Sungai

Pipeline end-to-end untuk audit data, feature engineering lingkungan dan HydroRIVERS, rolling-origin validation, ensemble XGBoost residual-klimatologi, dan pembuatan submission.

**Guard training:** notebook memakai `MODE = "audit"` dan `CONFIRM_TRAIN = False` secara default. Menjalankan seluruh cell tidak akan melakukan fitting. Desain dan hasil audit lengkap ada di `architecture.md` dan `logs.md`.

## 1. Import, path, dan konfigurasi model

In [13]:
"""Long-horizon river water-level forecasting with residual XGBoost.

Every future feature is exogenous, static, or calendar-derived. No target lag is
used, so the 242-day deployment horizon does not require recursive prediction.
Training is never started implicitly: MODE defaults to "audit" and training
modes additionally require CONFIRM_TRAIN = True.
"""

from __future__ import annotations

import importlib.metadata
import json
import logging
import platform
import re
import sys
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder

LOGGER = logging.getLogger("tma_xgb_residual")

@dataclass(frozen=True)
class Fold:
    name: str
    train_end: str
    valid_start: str
    valid_end: str


@dataclass
class PipelineConfig:
    """Single source of truth for feature, validation, and model settings."""

    data_dir: Path = Path("datasets")
    output_dir: Path = Path("artifacts")
    random_seeds: tuple[int, ...] = (17, 41, 83)
    upstream_decay_km: float = 75.0
    baseline_half_life_days: float = 365.0
    baseline_harmonics: int = 4
    min_station_labels: int = 90
    early_stopping_rounds: int = 250
    max_boost_rounds: int = 5000
    outlier_weight: float = 0.02
    robust_blend_grid: tuple[float, ...] = tuple(i / 10 for i in range(11))
    folds: tuple[Fold, ...] = field(
        default_factory=lambda: (
            Fold("sep_2023", "2023-09-18 18:00:00", "2023-09-19 06:00:00", "2024-05-18 18:00:00"),
            Fold("may_2024", "2024-05-18 18:00:00", "2024-05-19 06:00:00", "2025-01-18 18:00:00"),
            Fold("sep_2024", "2024-09-18 18:00:00", "2024-09-19 06:00:00", "2025-05-18 18:00:00"),
            Fold("jan_2025", "2025-01-18 18:00:00", "2025-01-19 06:00:00", "2025-09-18 18:00:00"),
        )
    )

    @property
    def train_path(self) -> Path:
        return self.data_dir / "train.csv"

    @property
    def test_path(self) -> Path:
        return self.data_dir / "test.csv"

    @property
    def sample_submission_path(self) -> Path:
        return self.data_dir / "sample_submission.csv"

    @property
    def environment_path(self) -> Path:
        return self.data_dir / "data_pendukung" / "data_lingkungan.csv"

    @property
    def stations_path(self) -> Path:
        return self.data_dir / "data_pendukung" / "koordinat_pos.csv"

    @property
    def rivers_path(self) -> Path:
        return self.data_dir / "data_pendukung" / "HydroRIVERS_v10_au_shp" / "HydroRIVERS_v10_au.shp"

    def resolve(self, root: Path) -> "PipelineConfig":
        self.data_dir = (root / self.data_dir).resolve() if not self.data_dir.is_absolute() else self.data_dir
        self.output_dir = (root / self.output_dir).resolve() if not self.output_dir.is_absolute() else self.output_dir
        return self

## 2. Parsing ID, pemuatan data, dan audit schema

ID diparse dengan regex agar nama pos yang mengandung tanda hubung tetap utuh. Audit menolak schema/key/coverage yang berubah dan tidak mengimputasi target yang tidak tercatat.

In [14]:
ID_PATTERN = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (.+)$")
TARGET_HOURS = {6, 12, 18}


@dataclass
class RawData:
    train: pd.DataFrame
    test: pd.DataFrame
    sample_submission: pd.DataFrame
    environment: pd.DataFrame
    stations: pd.DataFrame


def parse_test_ids(ids: pd.Series) -> pd.DataFrame:
    extracted = ids.astype(str).str.extract(ID_PATTERN)
    if extracted.isna().any().any():
        bad = ids[extracted.isna().any(axis=1)].head(5).tolist()
        raise ValueError(f"ID test tidak sesuai format timestamp - nama_pos: {bad}")
    extracted.columns = ["datetime", "nama_pos"]
    extracted["datetime"] = pd.to_datetime(extracted["datetime"], errors="raise")
    extracted["id"] = ids.to_numpy()
    return extracted[["id", "datetime", "nama_pos"]]


def load_raw_data(
    train_path: Path,
    test_path: Path,
    sample_submission_path: Path,
    environment_path: Path,
    stations_path: Path,
) -> RawData:
    train = pd.read_csv(train_path, parse_dates=["datetime"])
    test_ids = pd.read_csv(test_path)
    sample = pd.read_csv(sample_submission_path)
    environment = pd.read_csv(environment_path, parse_dates=["datetime"])
    stations = pd.read_csv(stations_path)
    test = parse_test_ids(test_ids["id"])
    validate_raw_data(train, test, sample, environment, stations)
    return RawData(train, test, sample, environment, stations)


def validate_raw_data(
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    environment: pd.DataFrame,
    stations: pd.DataFrame,
) -> None:
    required_train = {"datetime", "nama_pos", "tma_mdpl"}
    if not required_train.issubset(train):
        raise ValueError(f"Kolom train kurang: {required_train - set(train.columns)}")
    for name, frame in (("train", train), ("test", test), ("environment", environment)):
        if frame.duplicated(["datetime", "nama_pos"]).any():
            raise ValueError(f"Duplikat key (datetime, nama_pos) pada {name}")
    if train["tma_mdpl"].isna().any() or not np.isfinite(train["tma_mdpl"]).all():
        raise ValueError("Target train harus finite; baris target hilang tidak boleh diimputasi")
    if set(train["datetime"].dt.hour) - TARGET_HOURS:
        raise ValueError("Train berisi jam selain 06, 12, 18")
    if set(test["datetime"].dt.hour) - TARGET_HOURS:
        raise ValueError("Test berisi jam selain 06, 12, 18")
    if sample["id"].tolist() != test["id"].tolist():
        raise ValueError("Urutan ID test dan sample_submission tidak identik")
    station_sets = {
        "train": set(train["nama_pos"]),
        "test": set(test["nama_pos"]),
        "environment": set(environment["nama_pos"]),
        "coordinates": set(stations["nama_pos"]),
    }
    if len({frozenset(v) for v in station_sets.values()}) != 1:
        raise ValueError(f"Set nama pos tidak konsisten: { {k: len(v) for k, v in station_sets.items()} }")
    if environment["datetime"].min() > train["datetime"].min():
        raise ValueError("Data lingkungan tidak mencakup awal train")
    if environment["datetime"].max() < test["datetime"].max():
        raise ValueError("Data lingkungan tidak mencakup akhir test")


def make_data_audit(raw: RawData) -> dict[str, object]:
    env = raw.environment
    theoretical = pd.MultiIndex.from_product(
        [pd.date_range(env["datetime"].min(), env["datetime"].max(), freq="h"), sorted(env["nama_pos"].unique())],
        names=["datetime", "nama_pos"],
    )
    return {
        "train_rows": len(raw.train),
        "test_rows": len(raw.test),
        "environment_rows": len(env),
        "station_count": raw.train["nama_pos"].nunique(),
        "train_start": str(raw.train["datetime"].min()),
        "train_end": str(raw.train["datetime"].max()),
        "test_start": str(raw.test["datetime"].min()),
        "test_end": str(raw.test["datetime"].max()),
        "environment_grid_complete": len(theoretical) == len(env),
        "environment_missing_by_column": env.isna().sum().loc[lambda s: s.gt(0)].to_dict(),
        "solar_minus_999": int(env["solar_radiation_mj_m2"].eq(-999).sum()),
        "rainfall_sources_identical": bool(
            np.allclose(env["rainfall_mm"], env["rainfall_openmeteo_mm"], equal_nan=True)
        ),
    }

## 3. HydroRIVERS dan graf upstream

Pos dipasangkan ke reach terdekat dalam UTM 49S. `NEXT_DOWN` dipakai untuk traversal terarah dan bobot upstream meluruh terhadap jarak hidraulik.

In [15]:
RIVER_COLUMNS = [
    "HYRIV_ID",
    "NEXT_DOWN",
    "MAIN_RIV",
    "LENGTH_KM",
    "DIST_DN_KM",
    "DIST_UP_KM",
    "CATCH_SKM",
    "UPLAND_SKM",
    "ENDORHEIC",
    "DIS_AV_CMS",
    "ORD_STRA",
    "ORD_CLAS",
    "ORD_FLOW",
    "HYBAS_L12",
]


@dataclass
class HydroFeatures:
    station_static: pd.DataFrame
    upstream_weights: pd.DataFrame


def build_hydro_features(
    stations: pd.DataFrame,
    rivers_path: Path,
    decay_km: float = 75.0,
    bbox_padding_deg: float = 0.5,
) -> HydroFeatures:
    """Match stations to river reaches and derive directed upstream weights.

    `NEXT_DOWN` defines exact graph navigation. A source station contributes to a
    target station only when the target reach occurs on the source's downstream
    path. Self-contribution is always retained.
    """
    import geopandas as gpd

    station_gdf = gpd.GeoDataFrame(
        stations.copy(),
        geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]),
        crs="EPSG:4326",
    )
    minx, miny, maxx, maxy = station_gdf.total_bounds
    bbox = (minx - bbox_padding_deg, miny - bbox_padding_deg, maxx + bbox_padding_deg, maxy + bbox_padding_deg)
    rivers = gpd.read_file(rivers_path, bbox=bbox)
    missing = set(RIVER_COLUMNS) - set(rivers.columns)
    if missing:
        raise ValueError(f"Atribut HydroRIVERS kurang: {sorted(missing)}")
    utm = station_gdf.estimate_utm_crs()
    station_utm = station_gdf.to_crs(utm)
    rivers_utm = rivers.to_crs(utm)
    matched = gpd.sjoin_nearest(
        station_utm,
        rivers_utm[RIVER_COLUMNS + ["geometry"]],
        how="left",
        distance_col="river_distance_m",
    )
    matched = matched.sort_values("river_distance_m").drop_duplicates("nama_pos", keep="first")
    if matched["HYRIV_ID"].isna().any():
        raise ValueError("Ada pos yang tidak dapat dipasangkan ke HydroRIVERS")

    static = pd.DataFrame(matched.drop(columns=["geometry", "index_right"]))
    static["HYRIV_ID"] = static["HYRIV_ID"].astype(np.int64)
    static["river_match_confidence"] = np.exp(-static["river_distance_m"] / 1000.0)
    for src, dst in (
        ("CATCH_SKM", "log1p_catch_skm"),
        ("UPLAND_SKM", "log1p_upland_skm"),
        ("DIS_AV_CMS", "log1p_dis_av_cms"),
        ("LENGTH_KM", "log1p_length_km"),
    ):
        static[dst] = np.log1p(static[src].clip(lower=0))
    static["main_riv_cat"] = static["MAIN_RIV"].astype(str)
    static["hybas_l12_cat"] = static["HYBAS_L12"].astype(str)
    static["endorheic_cat"] = static["ENDORHEIC"].astype(str)

    # Use the local river subset: every path between nearby Java stations remains
    # within the padded subset. Stop safely if a path leaves it.
    river_index = rivers.set_index("HYRIV_ID")
    next_down = river_index["NEXT_DOWN"].astype(np.int64).to_dict()
    dist_down = river_index["DIST_DN_KM"].to_dict()
    station_reach = static.set_index("nama_pos")["HYRIV_ID"].astype(np.int64).to_dict()
    reach_to_stations: dict[int, list[str]] = {}
    for station, reach in station_reach.items():
        reach_to_stations.setdefault(reach, []).append(station)

    names = sorted(station_reach)
    weights = pd.DataFrame(0.0, index=names, columns=names)
    max_steps = len(next_down) + 1
    for source in names:
        source_reach = station_reach[source]
        current = source_reach
        visited: set[int] = set()
        steps = 0
        while current and current in next_down and current not in visited:
            visited.add(current)
            for target in reach_to_stations.get(current, []):
                distance = max(float(dist_down[source_reach]) - float(dist_down[current]), 0.0)
                weights.loc[target, source] = max(weights.loc[target, source], np.exp(-distance / decay_km))
            current = int(next_down[current])
            steps += 1
            if steps > max_steps:
                raise RuntimeError("Siklus tidak wajar ditemukan pada graph HydroRIVERS")
        weights.loc[source, source] = max(weights.loc[source, source], 1.0)

    row_sums = weights.sum(axis=1)
    if row_sums.eq(0).any():
        raise RuntimeError("Bobot upstream kosong untuk setidaknya satu pos")
    weights = weights.div(row_sums, axis=0)
    weights.index.name = "target_station"
    weights.columns.name = "source_station"
    return HydroFeatures(static.reset_index(drop=True), weights)


def upstream_weighted_series(
    environment: pd.DataFrame,
    weights: pd.DataFrame,
    source_columns: tuple[str, ...] = ("rainfall_mm", "soil_moisture_0_7cm"),
) -> pd.DataFrame:
    """Matrix-multiply station values into graph-aware upstream summaries."""
    pieces: list[pd.DataFrame] = []
    station_order = list(weights.columns)
    target_order = list(weights.index)
    matrix = weights.to_numpy(dtype=np.float64)
    for column in source_columns:
        wide = environment.pivot(index="datetime", columns="nama_pos", values=column).reindex(columns=station_order)
        values = wide.to_numpy(dtype=np.float64)
        upstream = values @ matrix.T
        frame = pd.DataFrame(upstream, index=wide.index, columns=target_order)
        long = (
            frame.rename_axis(index="datetime", columns="nama_pos")
            .reset_index()
            .melt(id_vars="datetime", var_name="nama_pos", value_name=f"upstream_{column}")
        )
        pieces.append(long)
    result = pieces[0]
    for piece in pieces[1:]:
        result = result.merge(piece, on=["datetime", "nama_pos"], how="inner", validate="one_to_one")
    return result

## 4. Sanitasi lingkungan dan fitur antecedent

Missing slow variables diisi secara kausal dengan batas waktu. Solar dikeluarkan dari model karena missing struktural pada test; rolling/API tidak pernah memakai masa depan.

In [16]:
KEYS = ["datetime", "nama_pos"]

SLOW_24H = (
    "soil_moisture_0_7cm",
    "soil_moisture_7_28cm",
    "soil_moisture_28_100cm",
    "soil_moisture_100_255cm",
    "surface_pressure_hpa",
    "pressure_msl_hpa",
    "rmm1",
    "rmm2",
    "mjo_phase",
    "mjo_amplitude",
    "mjo_active",
)

CATEGORICAL_FEATURES = (
    "nama_pos",
    "landcover_name",
    "hour_cat",
    "month_cat",
    "mjo_phase_cat",
    "main_riv_cat",
    "hybas_l12_cat",
    "endorheic_cat",
    "landcover_class_cat",
)

EXCLUDED_FROM_MODEL = {
    "datetime",
    "id",
    "tma_mdpl",
    "rainfall_openmeteo_mm",  # exact duplicate
    "solar_radiation_mj_m2",  # structurally absent in 57.4% of test
    "solar_radiation_missing",  # would identify the forecast source regime
    "wind_direction_deg",
    "mjo_phase",
    "landcover_class",
    "HYRIV_ID",
    "NEXT_DOWN",
    "MAIN_RIV",
    "HYBAS_L12",
}


def sanitize_environment(environment: pd.DataFrame) -> pd.DataFrame:
    env = environment.copy().sort_values(KEYS).reset_index(drop=True)
    if not np.allclose(env["rainfall_mm"], env["rainfall_openmeteo_mm"], equal_nan=True):
        raise ValueError("Dua sumber hujan tidak lagi identik; keputusan deduplikasi perlu ditinjau")
    env["solar_radiation_mj_m2"] = env["solar_radiation_mj_m2"].mask(
        env["solar_radiation_mj_m2"].eq(-999)
    )
    tracked_missing = ("solar_radiation_mj_m2",) + SLOW_24H + ("nino_34",)
    for column in tracked_missing:
        env[f"{column.removesuffix('_mj_m2')}_missing"] = env[column].isna().astype(np.int8)

    grouped = env.groupby("nama_pos", sort=False)
    for column in SLOW_24H:
        env[column] = grouped[column].ffill(limit=24)
    # Nino 3.4 is monthly/slow-moving; 31 days is a causal and documented cap.
    env["nino_34"] = grouped["nino_34"].ffill(limit=31 * 24)

    env["rainfall_mm"] = env["rainfall_mm"].clip(lower=0)
    env["rainfall_max_24h_mm"] = env["rainfall_max_24h_mm"].clip(lower=0)
    for column in [c for c in SLOW_24H if c.startswith("soil_moisture")]:
        env[column] = env[column].clip(0, 1)
    angle = np.deg2rad(env["wind_direction_deg"] % 360)
    env["wind_direction_sin"] = np.sin(angle)
    env["wind_direction_cos"] = np.cos(angle)
    env["dewpoint_depression_c"] = env["temperature_c"] - env["dew_point_c"]
    phase_angle = 2 * np.pi * ((env["mjo_phase"] - 1) % 8) / 8
    env["mjo_phase_sin"] = np.sin(phase_angle)
    env["mjo_phase_cos"] = np.cos(phase_angle)
    return env


def _rolling_feature(
    frame: pd.DataFrame,
    source: str,
    window: int,
    operation: str,
    output: str,
) -> None:
    grouped = frame.groupby("nama_pos", sort=False)[source]
    if operation == "sum":
        frame[output] = grouped.transform(lambda x: x.rolling(window, min_periods=window).sum())
    elif operation == "mean":
        frame[output] = grouped.transform(lambda x: x.rolling(window, min_periods=window).mean())
    elif operation == "max":
        frame[output] = grouped.transform(lambda x: x.rolling(window, min_periods=window).max())
    else:
        raise ValueError(f"Operasi rolling tidak dikenal: {operation}")


def _ewm_feature(frame: pd.DataFrame, source: str, half_life: int, output: str) -> None:
    frame[output] = frame.groupby("nama_pos", sort=False)[source].transform(
        lambda x: x.ewm(halflife=half_life, adjust=False, min_periods=1).mean()
    )


def add_dynamic_features(environment: pd.DataFrame, hydro: HydroFeatures) -> pd.DataFrame:
    env = sanitize_environment(environment)
    upstream = upstream_weighted_series(env, hydro.upstream_weights)
    env = env.merge(upstream, on=KEYS, how="left", validate="one_to_one")

    for source in ("rainfall_mm", "upstream_rainfall_mm"):
        for window in (6, 24, 72, 168, 336):
            _rolling_feature(env, source, window, "sum", f"{source}_sum_{window}h")
        _rolling_feature(env, source, 24, "max", f"{source}_max_24h")
        for half_life in (12, 48, 168):
            _ewm_feature(env, source, half_life, f"{source}_api_halflife_{half_life}h")

    for source in (
        "soil_moisture_0_7cm",
        "soil_moisture_28_100cm",
        "upstream_soil_moisture_0_7cm",
    ):
        for window in (24, 168):
            _rolling_feature(env, source, window, "mean", f"{source}_mean_{window}h")
    for source in ("temperature_c", "humidity_pct"):
        for window in (24, 72):
            _rolling_feature(env, source, window, "mean", f"{source}_mean_{window}h")
    _rolling_feature(env, "surface_pressure_hpa", 24, "mean", "surface_pressure_hpa_mean_24h")
    return env

## 5. Kalender, kategori, dan matriks fitur

In [17]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    dt = result["datetime"]
    day_angle = 2 * np.pi * (dt.dt.dayofyear - 1) / 365.2425
    hour_angle = 2 * np.pi * dt.dt.hour / 24
    result["dayofyear_sin"] = np.sin(day_angle)
    result["dayofyear_cos"] = np.cos(day_angle)
    result["hour_sin"] = np.sin(hour_angle)
    result["hour_cos"] = np.cos(hour_angle)
    result["is_wet_season"] = dt.dt.month.isin([11, 12, 1, 2, 3, 4]).astype(np.int8)
    result["hour_cat"] = dt.dt.hour.astype(str)
    result["month_cat"] = dt.dt.month.astype(str)
    phase = result["mjo_phase"].round().astype("Int64")
    result["mjo_phase_cat"] = phase.astype(str)
    result["landcover_class_cat"] = result["landcover_class"].astype(str)
    return result


def assemble_feature_matrices(
    train: pd.DataFrame,
    test: pd.DataFrame,
    environment: pd.DataFrame,
    hydro: HydroFeatures,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    dynamic = add_dynamic_features(environment, hydro)
    static = hydro.station_static.copy()
    dynamic = dynamic.merge(static, on="nama_pos", how="left", validate="many_to_one")
    dynamic = add_calendar_features(dynamic)

    train_features = train.merge(dynamic, on=KEYS, how="left", validate="one_to_one")
    test_features = test.merge(dynamic, on=KEYS, how="left", validate="one_to_one")
    if train_features["rainfall_mm"].isna().any() or test_features["rainfall_mm"].isna().any():
        raise ValueError("Merge lingkungan gagal untuk sebagian baris train/test")

    feature_columns = [c for c in train_features.columns if c not in EXCLUDED_FROM_MODEL]
    categorical = [c for c in CATEGORICAL_FEATURES if c in feature_columns]
    forbidden = [c for c in feature_columns if c.startswith("tma_") or "target" in c.lower()]
    if forbidden:
        raise RuntimeError(f"Potensi target leakage pada feature list: {forbidden}")
    return train_features, test_features, feature_columns, categorical

## 6. Baseline harmonik, target residual, spike weighting, metrik, dan XGBoost

Baseline Ridge per pos menangani level/musim. XGBoost mempelajari residual robust-scaled dengan ensemble squared-error dan pseudo-Huber pada tiga seed.

In [18]:
def _harmonic_matrix(datetime: pd.Series, harmonics: int) -> np.ndarray:
    dt = pd.to_datetime(datetime)
    day = (dt.dt.dayofyear.to_numpy(dtype=float) - 1.0) / 365.2425
    hour = dt.dt.hour.to_numpy(dtype=float) / 24.0
    columns: list[np.ndarray] = []
    for k in range(1, harmonics + 1):
        columns.extend((np.sin(2 * np.pi * k * day), np.cos(2 * np.pi * k * day)))
    hour_sin = np.sin(2 * np.pi * hour)
    hour_cos = np.cos(2 * np.pi * hour)
    columns.extend((hour_sin, hour_cos))
    # Season-hour interactions let the 06/12/18 offset vary smoothly by season.
    columns.extend((columns[0] * hour_sin, columns[0] * hour_cos, columns[1] * hour_sin, columns[1] * hour_cos))
    return np.column_stack(columns)


def compute_label_weights(
    train: pd.DataFrame,
    downweight: float = 0.02,
) -> pd.DataFrame:
    """Conservatively downweight isolated sensor spikes without deleting floods.

    A point is suspicious only when both adjacent observations are temporally
    close, agree with one another, and the current value makes a very large jump
    relative to both station level variability and normal step variability.
    Multi-timestamp high-water episodes therefore remain full weight.
    """
    result = train[["datetime", "nama_pos", "tma_mdpl"]].copy()
    result["_order"] = np.arange(len(result))
    result = result.sort_values(["nama_pos", "datetime"])
    grouped = result.groupby("nama_pos", sort=False)
    result["prev"] = grouped["tma_mdpl"].shift(1)
    result["next"] = grouped["tma_mdpl"].shift(-1)
    result["prev_time"] = grouped["datetime"].shift(1)
    result["next_time"] = grouped["datetime"].shift(-1)

    station_median = grouped["tma_mdpl"].transform("median")
    abs_dev = (result["tma_mdpl"] - station_median).abs()
    level_mad = abs_dev.groupby(result["nama_pos"]).transform("median")
    step = grouped["tma_mdpl"].diff().abs()
    # Lower quartile estimates ordinary sensor movement and is not inflated by
    # a short real flood plateau or by the two edges of one corrupt spike.
    step_mad_map = step.groupby(result["nama_pos"]).quantile(0.25)
    step_scale = result["nama_pos"].map(step_mad_map).fillna(0.0).clip(lower=0.05) * 1.4826
    level_scale = level_mad.clip(lower=0.05) * 1.4826

    neighbors_close_in_time = (
        result["datetime"].sub(result["prev_time"]).le(pd.Timedelta(hours=12))
        & result["next_time"].sub(result["datetime"]).le(pd.Timedelta(hours=12))
    )
    neighbor_gap = (result["prev"] - result["next"]).abs()
    neighbors_agree = neighbor_gap.le(np.maximum(8 * step_scale, 1.0))
    neighbor_center = (result["prev"] + result["next"]) / 2
    jump_threshold = np.maximum.reduce([8 * step_scale, 5 * level_scale, np.full(len(result), 5.0)])
    jump = (result["tma_mdpl"] - neighbor_center).abs()
    extreme_isolated = (
        neighbors_close_in_time
        & neighbor_gap.le(np.maximum(30 * step_scale, 3.0))
        & jump.gt(np.maximum(20 * level_scale, 20.0))
    )
    previous_close = result["datetime"].sub(result["prev_time"]).le(pd.Timedelta(hours=12))
    next_close = result["next_time"].sub(result["datetime"]).le(pd.Timedelta(hours=12))
    one_sided_jump = pd.concat(
        [
            (result["tma_mdpl"] - result["prev"]).abs().fillna(np.inf),
            (result["tma_mdpl"] - result["next"]).abs().fillna(np.inf),
        ],
        axis=1,
    ).min(axis=1)
    one_sided_extreme = (
        (previous_close | next_close)
        & abs_dev.gt(np.maximum(50 * level_scale, 20.0))
        & one_sided_jump.gt(np.maximum(50 * step_scale, 20.0))
    )
    isolated = (
        (neighbors_close_in_time & neighbors_agree & jump.gt(jump_threshold))
        | extreme_isolated
        | one_sided_extreme
    )

    result["label_anomaly_flag"] = isolated.astype(np.int8)
    result["sample_weight"] = np.where(isolated, downweight, 1.0)
    return result.sort_values("_order")[["sample_weight", "label_anomaly_flag"]].reset_index(drop=True)


@dataclass
class StationHarmonicBaseline:
    harmonics: int = 4
    half_life_days: float = 365.0
    ridge_alpha: float = 8.0
    models: dict[str, Ridge] = field(default_factory=dict, init=False)
    fallback: Ridge | None = field(default=None, init=False)

    def fit(self, frame: pd.DataFrame, sample_weight: np.ndarray | pd.Series | None = None) -> "StationHarmonicBaseline":
        if sample_weight is None:
            base_weight = np.ones(len(frame), dtype=float)
        else:
            base_weight = np.asarray(sample_weight, dtype=float)
        newest = frame["datetime"].max()
        age_days = (newest - frame["datetime"]).dt.total_seconds().to_numpy() / 86400.0
        recency = np.exp(-np.log(2.0) * age_days / self.half_life_days)
        recency = np.clip(recency, 0.05, 1.0)
        weights = base_weight * recency

        # Fallback is only for defensive handling; all 30 deployment stations are seen.
        x_all = _harmonic_matrix(frame["datetime"], self.harmonics)
        self.fallback = Ridge(alpha=self.ridge_alpha).fit(x_all, frame["tma_mdpl"], sample_weight=weights)
        self.models = {}
        for station, idx in frame.groupby("nama_pos", sort=False).groups.items():
            idx_array = frame.index.get_indexer(idx)
            x = _harmonic_matrix(frame.loc[idx, "datetime"], self.harmonics)
            model = Ridge(alpha=self.ridge_alpha)
            model.fit(x, frame.loc[idx, "tma_mdpl"], sample_weight=weights[idx_array])
            self.models[str(station)] = model
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        if self.fallback is None:
            raise RuntimeError("Baseline belum di-fit")
        prediction = np.empty(len(frame), dtype=float)
        for station, positions in frame.groupby("nama_pos", sort=False).indices.items():
            model = self.models.get(str(station), self.fallback)
            subset = frame.iloc[positions]
            prediction[positions] = model.predict(_harmonic_matrix(subset["datetime"], self.harmonics))
        return prediction


def station_residual_scales(frame: pd.DataFrame, residual: np.ndarray) -> pd.Series:
    tmp = pd.DataFrame({"nama_pos": frame["nama_pos"].to_numpy(), "residual": residual})
    q = tmp.groupby("nama_pos")["residual"].quantile([0.25, 0.75]).unstack()
    scale = ((q[0.75] - q[0.25]) / 1.349).clip(lower=0.05)
    return scale


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    error = np.asarray(y_pred) - np.asarray(y_true)
    absolute = np.abs(error)
    return {
        "rmse": float(np.sqrt(np.mean(error**2))),
        "mae": float(np.mean(absolute)),
        "bias": float(np.mean(error)),
        "p95_absolute_error": float(np.quantile(absolute, 0.95)),
    }


def choose_blend_weight(
    y_true: np.ndarray,
    squared_prediction: np.ndarray,
    robust_prediction: np.ndarray,
    clean_mask: np.ndarray,
    grid: tuple[float, ...],
) -> tuple[float, pd.DataFrame]:
    """Select pseudo-Huber share using clean RMSE plus a raw-RMSE guardrail."""
    rows = []
    for robust_weight in grid:
        prediction = (1 - robust_weight) * squared_prediction + robust_weight * robust_prediction
        raw_rmse = regression_metrics(y_true, prediction)["rmse"]
        clean_rmse = regression_metrics(y_true[clean_mask], prediction[clean_mask])["rmse"]
        rows.append(
            {
                "robust_weight": robust_weight,
                "raw_rmse": raw_rmse,
                "clean_rmse": clean_rmse,
                "selection_score": clean_rmse + 0.25 * raw_rmse,
            }
        )
    scores = pd.DataFrame(rows)
    best = float(scores.sort_values(["selection_score", "raw_rmse"]).iloc[0]["robust_weight"])
    return best, scores


@dataclass
class XGBResidualEnsemble:
    feature_columns: list[str]
    categorical_columns: list[str]
    seeds: tuple[int, ...] = (17, 41, 83)
    max_rounds: int = 5000
    early_stopping_rounds: int = 250
    preprocessor: ColumnTransformer | None = field(default=None, init=False)
    models: dict[str, list[Any]] = field(default_factory=lambda: {"squared": [], "robust": []}, init=False)
    best_iterations: list[dict[str, int | str]] = field(default_factory=list, init=False)

    @property
    def numeric_columns(self) -> list[str]:
        return [c for c in self.feature_columns if c not in self.categorical_columns]

    def _clean_frame(self, frame: pd.DataFrame) -> pd.DataFrame:
        result = frame[self.feature_columns].copy()
        for column in self.categorical_columns:
            result[column] = result[column].astype("string").fillna("__MISSING__")
        for column in self.numeric_columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
        return result

    def _make_preprocessor(self) -> ColumnTransformer:
        return ColumnTransformer(
            transformers=[
                (
                    "categorical",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32),
                    self.categorical_columns,
                ),
                ("numeric", "passthrough", self.numeric_columns),
            ],
            sparse_threshold=1.0,
            verbose_feature_names_out=False,
        )

    def fit(
        self,
        train_frame: pd.DataFrame,
        y_train: np.ndarray,
        valid_frame: pd.DataFrame | None = None,
        y_valid: np.ndarray | None = None,
        sample_weight: np.ndarray | None = None,
        rounds_override: int | dict[str, int] | None = None,
    ) -> "XGBResidualEnsemble":
        from xgboost import XGBRegressor

        self.preprocessor = self._make_preprocessor()
        x_train = self.preprocessor.fit_transform(self._clean_frame(train_frame))
        x_valid = None if valid_frame is None else self.preprocessor.transform(self._clean_frame(valid_frame))
        self.models = {"squared": [], "robust": []}
        self.best_iterations = []
        objectives = {"squared": "reg:squarederror", "robust": "reg:pseudohubererror"}
        for family, objective in objectives.items():
            for seed in self.seeds:
                if isinstance(rounds_override, dict):
                    rounds = int(rounds_override[family])
                else:
                    rounds = int(rounds_override or self.max_rounds)
                params: dict[str, Any] = {
                    "objective": objective,
                    "n_estimators": rounds,
                    "learning_rate": 0.025,
                    "max_depth": 7,
                    "min_child_weight": 20,
                    "subsample": 0.85,
                    "colsample_bytree": 0.80,
                    "reg_alpha": 0.10,
                    "reg_lambda": 20.0,
                    "max_bin": 256,
                    "tree_method": "hist",  # Tetap hist
                    "device": "cuda",       # Memerintahkan XGBoost untuk pindah ke GPU
                    "eval_metric": "rmse",
                    "random_state": seed,
                    "verbosity": 1,
                }
                if objective == "reg:pseudohubererror":
                    params["huber_slope"] = 1.0
                if x_valid is not None:
                    params["early_stopping_rounds"] = self.early_stopping_rounds
                model = XGBRegressor(**params)
                fit_kwargs: dict[str, Any] = {"sample_weight": sample_weight, "verbose": False}
                if x_valid is not None and y_valid is not None:
                    fit_kwargs["eval_set"] = [(x_valid, y_valid)]
                model.fit(x_train, y_train, **fit_kwargs)
                self.models[family].append(model)
                best = getattr(model, "best_iteration", rounds - 1)
                self.best_iterations.append({"family": family, "seed": seed, "best_iteration": int(best)})
        return self

    def predict_components(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        if self.preprocessor is None or not all(self.models.values()):
            raise RuntimeError("Ensemble belum di-fit")
        x = self.preprocessor.transform(self._clean_frame(frame))
        squared = np.mean([model.predict(x) for model in self.models["squared"]], axis=0)
        robust = np.mean([model.predict(x) for model in self.models["robust"]], axis=0)
        return squared, robust

    def encoded_feature_names(self) -> list[str]:
        if self.preprocessor is None:
            raise RuntimeError("Preprocessor belum di-fit")
        return self.preprocessor.get_feature_names_out().tolist()


@dataclass
class ForecastBundle:
    baseline: Any
    residual_scales: pd.Series
    ensemble: XGBResidualEnsemble
    robust_weight: float
    train_minimum: pd.Series
    train_maximum: pd.Series
    metadata: dict[str, Any]

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        base = self.baseline.predict(frame)
        squared, robust = self.ensemble.predict_components(frame)
        normalized = (1 - self.robust_weight) * squared + self.robust_weight * robust
        scale = frame["nama_pos"].map(self.residual_scales).to_numpy(dtype=float)
        prediction = base + normalized * scale
        if not np.isfinite(prediction).all():
            raise RuntimeError("Prediksi mengandung NaN/Inf")
        # Wide, station-specific safety rails only catch catastrophic extrapolation.
        lower = frame["nama_pos"].map(self.train_minimum).to_numpy(dtype=float) - 5 * scale
        upper = frame["nama_pos"].map(self.train_maximum).to_numpy(dtype=float) + 5 * scale
        return np.clip(prediction, lower, upper)

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(self, path, compress=3)

    @staticmethod
    def load(path: Path) -> "ForecastBundle":
        return joblib.load(path)

## 7. Run tracking dan artefak untuk penyempurnaan model

Audit, exact feature matrices, graph weights, feature list, log, OOF, metrik, model bundle, dan submission disimpan di `artifacts/` atau `/kaggle/working`.

In [19]:
LOGGER = logging.getLogger(__name__)


@dataclass
class PreparedData:
    raw: RawData
    hydro: HydroFeatures
    train_features: pd.DataFrame
    test_features: pd.DataFrame
    feature_columns: list[str]
    categorical_columns: list[str]


def _write_json(value: object, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")


def load_and_audit(config: PipelineConfig) -> RawData:
    raw = load_raw_data(
        config.train_path,
        config.test_path,
        config.sample_submission_path,
        config.environment_path,
        config.stations_path,
    )
    audit = make_data_audit(raw)
    _write_json(audit, config.output_dir / "audit" / "data_audit.json")
    LOGGER.info("Audit selesai: %s", audit)
    return raw


def prepare_features(config: PipelineConfig) -> PreparedData:
    raw = load_and_audit(config)
    hydro = build_hydro_features(raw.stations, config.rivers_path, config.upstream_decay_km)
    train_features, test_features, feature_columns, categorical = assemble_feature_matrices(
        raw.train, raw.test, raw.environment, hydro
    )
    prepared_dir = config.output_dir / "prepared"
    prepared_dir.mkdir(parents=True, exist_ok=True)
    train_features.to_parquet(prepared_dir / "train_features.parquet", index=False)
    test_features.to_parquet(prepared_dir / "test_features.parquet", index=False)
    hydro.station_static.to_csv(prepared_dir / "station_hydro_features.csv", index=False)
    hydro.upstream_weights.to_csv(prepared_dir / "upstream_weights.csv")
    _write_json(
        {"features": feature_columns, "categorical": categorical},
        prepared_dir / "feature_columns.json",
    )
    return PreparedData(raw, hydro, train_features, test_features, feature_columns, categorical)


def _recency_weights(datetime: pd.Series, half_life_days: float = 730.0) -> np.ndarray:
    age = (datetime.max() - datetime).dt.total_seconds().to_numpy() / 86400.0
    return np.clip(np.exp(-np.log(2.0) * age / half_life_days), 0.25, 1.0)


def _make_ensemble(prepared: PreparedData, config: PipelineConfig) -> XGBResidualEnsemble:
    return XGBResidualEnsemble(
        prepared.feature_columns,
        prepared.categorical_columns,
        seeds=config.random_seeds,
        max_rounds=config.max_boost_rounds,
        early_stopping_rounds=config.early_stopping_rounds,
    )

## 8. Rolling-origin validation

Empat fold meniru horizon panjang deployment. Baseline, residual scale, encoder, dan model selalu di-fit ulang hanya pada prefix train fold.

In [20]:
def validate(prepared: PreparedData, config: PipelineConfig) -> dict[str, object]:
    train = prepared.train_features
    oof_parts: list[pd.DataFrame] = []
    iteration_rows: list[dict[str, object]] = []
    for fold in config.folds:
        train_mask = train["datetime"].le(pd.Timestamp(fold.train_end))
        valid_mask = train["datetime"].between(pd.Timestamp(fold.valid_start), pd.Timestamp(fold.valid_end))
        fold_train = train.loc[train_mask].copy()
        counts = fold_train.groupby("nama_pos").size()
        eligible = set(counts[counts.ge(config.min_station_labels)].index)
        fold_valid = train.loc[valid_mask & train["nama_pos"].isin(eligible)].copy()
        if fold_train.empty or fold_valid.empty:
            raise RuntimeError(f"Fold {fold.name} kosong")

        train_quality = compute_label_weights(fold_train, config.outlier_weight)
        baseline = StationHarmonicBaseline(
            harmonics=config.baseline_harmonics,
            half_life_days=config.baseline_half_life_days,
        ).fit(fold_train, train_quality["sample_weight"])
        baseline_train = baseline.predict(fold_train)
        baseline_valid = baseline.predict(fold_valid)
        train_residual = fold_train["tma_mdpl"].to_numpy() - baseline_train
        scales = station_residual_scales(fold_train, train_residual)
        train_scale = fold_train["nama_pos"].map(scales).to_numpy(dtype=float)
        valid_scale = fold_valid["nama_pos"].map(scales).to_numpy(dtype=float)
        y_train = train_residual / train_scale
        y_valid = (fold_valid["tma_mdpl"].to_numpy() - baseline_valid) / valid_scale
        sample_weight = train_quality["sample_weight"].to_numpy() * _recency_weights(fold_train["datetime"])

        ensemble = _make_ensemble(prepared, config).fit(
            fold_train,
            y_train,
            valid_frame=fold_valid,
            y_valid=y_valid,
            sample_weight=sample_weight,
        )
        squared_norm, robust_norm = ensemble.predict_components(fold_valid)
        part = fold_valid[["datetime", "nama_pos", "tma_mdpl"]].copy()
        part["fold"] = fold.name
        part["baseline_prediction"] = baseline_valid
        part["squared_prediction"] = baseline_valid + squared_norm * valid_scale
        part["robust_prediction"] = baseline_valid + robust_norm * valid_scale
        valid_quality = compute_label_weights(fold_valid, config.outlier_weight)
        part["label_anomaly_flag"] = valid_quality["label_anomaly_flag"].to_numpy()
        oof_parts.append(part)
        for row in ensemble.best_iterations:
            iteration_rows.append({"fold": fold.name, **row})
        LOGGER.info("Fold %s selesai: train=%d valid=%d", fold.name, len(fold_train), len(fold_valid))

    oof = pd.concat(oof_parts, ignore_index=True)
    clean_mask = oof["label_anomaly_flag"].eq(0).to_numpy()
    robust_weight, blend_scores = choose_blend_weight(
        oof["tma_mdpl"].to_numpy(),
        oof["squared_prediction"].to_numpy(),
        oof["robust_prediction"].to_numpy(),
        clean_mask,
        config.robust_blend_grid,
    )
    oof["prediction"] = (
        (1 - robust_weight) * oof["squared_prediction"] + robust_weight * oof["robust_prediction"]
    )
    oof["error"] = oof["prediction"] - oof["tma_mdpl"]
    oof["absolute_error"] = oof["error"].abs()

    metric_rows: list[dict[str, object]] = []
    station_rows: list[dict[str, object]] = []
    for fold_name, group in oof.groupby("fold", sort=False):
        clean = group["label_anomaly_flag"].eq(0).to_numpy()
        raw_metrics = regression_metrics(group["tma_mdpl"].to_numpy(), group["prediction"].to_numpy())
        clean_metrics = regression_metrics(
            group.loc[clean, "tma_mdpl"].to_numpy(), group.loc[clean, "prediction"].to_numpy()
        )
        baseline_metrics = regression_metrics(
            group["tma_mdpl"].to_numpy(), group["baseline_prediction"].to_numpy()
        )
        station_mae = group.groupby("nama_pos")["absolute_error"].mean()
        metric_rows.append(
            {
                "fold": fold_name,
                "rows": len(group),
                "anomalies": int((~clean).sum()),
                "macro_station_mae": float(station_mae.mean()),
                "worst_station_mae": float(station_mae.max()),
                **{f"raw_{k}": v for k, v in raw_metrics.items()},
                **{f"clean_{k}": v for k, v in clean_metrics.items()},
                **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
            }
        )
        for station, station_group in group.groupby("nama_pos", sort=False):
            station_metric = regression_metrics(
                station_group["tma_mdpl"].to_numpy(), station_group["prediction"].to_numpy()
            )
            station_rows.append(
                {"fold": fold_name, "nama_pos": station, "rows": len(station_group), **station_metric}
            )
    metrics = pd.DataFrame(metric_rows)
    station_metrics = pd.DataFrame(station_rows)
    sliced = oof.assign(month=oof["datetime"].dt.month, hour=oof["datetime"].dt.hour)
    residual_slices = (
        sliced.groupby(["fold", "nama_pos", "month", "hour"], observed=True)
        .agg(
            rows=("error", "size"),
            mae=("absolute_error", "mean"),
            bias=("error", "mean"),
            p95_absolute_error=("absolute_error", lambda x: float(x.quantile(0.95))),
        )
        .reset_index()
    )
    iterations = pd.DataFrame(iteration_rows)
    recommended_rounds = (
        iterations.groupby("family")["best_iteration"].median().add(1).round().astype(int).to_dict()
    )
    validation_dir = config.output_dir / "validation"
    validation_dir.mkdir(parents=True, exist_ok=True)
    oof.to_parquet(validation_dir / "oof_predictions.parquet", index=False)
    metrics.to_csv(validation_dir / "metrics.csv", index=False)
    station_metrics.to_csv(validation_dir / "station_metrics.csv", index=False)
    residual_slices.to_csv(validation_dir / "residual_slices.csv", index=False)
    blend_scores.to_csv(validation_dir / "blend_search.csv", index=False)
    iterations.to_csv(validation_dir / "best_iterations.csv", index=False)
    summary = {
        "robust_weight": robust_weight,
        "recommended_rounds": recommended_rounds,
        "metrics": metrics.to_dict(orient="records"),
    }
    _write_json(summary, validation_dir / "summary.json")
    return summary

## 9. Final ensemble dan submission

Round final adalah median best iteration OOF per objective. Prediksi di-inverse-scale, ditambah baseline, diperiksa finite, lalu ditulis dalam urutan sample submission.

In [21]:
def fit_final(
    prepared: PreparedData,
    config: PipelineConfig,
    validation_summary: dict[str, object],
) -> ForecastBundle:
    train = prepared.train_features.copy()
    quality = compute_label_weights(train, config.outlier_weight)
    baseline = StationHarmonicBaseline(
        harmonics=config.baseline_harmonics,
        half_life_days=config.baseline_half_life_days,
    ).fit(train, quality["sample_weight"])
    baseline_train = baseline.predict(train)
    residual = train["tma_mdpl"].to_numpy() - baseline_train
    scales = station_residual_scales(train, residual)
    y = residual / train["nama_pos"].map(scales).to_numpy(dtype=float)
    sample_weight = quality["sample_weight"].to_numpy() * _recency_weights(train["datetime"])
    rounds = {k: int(v) for k, v in dict(validation_summary["recommended_rounds"]).items()}
    ensemble = _make_ensemble(prepared, config).fit(
        train,
        y,
        sample_weight=sample_weight,
        rounds_override=rounds,
    )
    target_bounds = train.groupby("nama_pos")["tma_mdpl"].quantile([0.001, 0.999]).unstack()
    bundle = ForecastBundle(
        baseline=baseline,
        residual_scales=scales,
        ensemble=ensemble,
        robust_weight=float(validation_summary["robust_weight"]),
        train_minimum=target_bounds[0.001],
        train_maximum=target_bounds[0.999],
        metadata={
            "config": asdict(config),
            "recommended_rounds": rounds,
            "feature_columns": prepared.feature_columns,
            "categorical_columns": prepared.categorical_columns,
            "isolated_labels_downweighted": int(quality["label_anomaly_flag"].sum()),
        },
    )
    model_dir = config.output_dir / "model"
    bundle.save(model_dir / "forecast_bundle.joblib")
    _write_json(bundle.metadata, model_dir / "metadata.json")
    feature_names = ensemble.encoded_feature_names()
    importance_rows = []
    for family, models in ensemble.models.items():
        for seed, model in zip(config.random_seeds, models, strict=True):
            for feature, importance in zip(feature_names, model.feature_importances_, strict=True):
                importance_rows.append(
                    {"family": family, "seed": seed, "feature": feature, "importance": float(importance)}
                )
    pd.DataFrame(importance_rows).to_csv(model_dir / "feature_importance.csv", index=False)
    return bundle


def write_submission(
    bundle: ForecastBundle,
    prepared: PreparedData,
    config: PipelineConfig,
) -> Path:
    prediction = bundle.predict(prepared.test_features)
    submission = prepared.raw.sample_submission[["id"]].copy()
    if submission["id"].tolist() != prepared.test_features["id"].tolist():
        raise RuntimeError("Urutan test berubah sebelum pembuatan submission")
    submission["tma_mdpl"] = prediction
    if submission["tma_mdpl"].isna().any() or not np.isfinite(submission["tma_mdpl"]).all():
        raise RuntimeError("Submission mengandung nilai non-finite")
    path = config.output_dir / "submission.csv"
    submission.to_csv(path, index=False)
    return path

## 10. Orkestrator notebook

In [22]:
def run_pipeline(
    mode: str = "audit",
    data_dir: Path = Path("datasets"),
    output_dir: Path = Path("artifacts"),
    save_feature_matrices: bool = True,
) -> dict[str, Any]:
    """Execute one notebook mode with audit as the safe default."""
    allowed = {"audit", "prepare", "validate", "fit", "all", "predict"}
    if mode not in allowed:
        raise ValueError(f"mode harus salah satu dari {sorted(allowed)}")
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        force=True,
    )
    config = PipelineConfig(data_dir=Path(data_dir), output_dir=Path(output_dir))
    config.output_dir.mkdir(parents=True, exist_ok=True)

    if mode == "audit":
        raw = load_and_audit(config)
        return {"mode": mode, "audit": make_data_audit(raw)}

    prepared = prepare_features(config)
    if not save_feature_matrices:
        LOGGER.warning("SAVE_FEATURE_MATRICES=False diabaikan; matriks tetap disimpan untuk reproducibility")
    result: dict[str, Any] = {
        "mode": mode,
        "train_rows": len(prepared.train_features),
        "test_rows": len(prepared.test_features),
        "feature_count": len(prepared.feature_columns),
    }
    if mode == "prepare":
        return result
    if mode == "validate":
        result["validation"] = validate(prepared, config)
        return result
    summary_path = config.output_dir / "validation" / "summary.json"
    if mode == "fit":
        if not summary_path.exists():
            raise FileNotFoundError("Jalankan MODE='validate' terlebih dahulu")
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        fit_final(prepared, config, summary)
        result["model"] = str(config.output_dir / "model" / "forecast_bundle.joblib")
        return result
    if mode == "all":
        summary = validate(prepared, config)
        bundle = fit_final(prepared, config, summary)
        result["validation"] = summary
        result["submission"] = str(write_submission(bundle, prepared, config))
        return result
    bundle = ForecastBundle.load(config.output_dir / "model" / "forecast_bundle.joblib")
    result["submission"] = str(write_submission(bundle, prepared, config))
    return result

## 11. Pilih mode eksekusi

- `audit`: audit input saja; aman untuk Run All.
- `prepare`: audit dan feature engineering tanpa training.
- `validate`: rolling-origin training/OOF.
- `fit`: final fit memakai hasil validasi yang sudah ada.
- `all`: validasi, final fit, dan submission.
- `predict`: inference dari model bundle yang sudah ada.

Untuk `validate`, `fit`, atau `all`, ubah juga `CONFIRM_TRAIN = True`.

In [23]:
# Guard: Run All hanya melakukan audit dan TIDAK melatih model.
KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026")
ON_KAGGLE = KAGGLE_DATA_DIR.exists()

MODE = "validate"  # pilihan: audit, prepare, validate, fit, all, predict
CONFIRM_TRAIN = True  # wajib True untuk validate/fit/all
DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else Path("datasets")
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")
SAVE_FEATURE_MATRICES = True

print(f"Runtime: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Input : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Mode  : {MODE}")

Runtime: Kaggle
Input : /kaggle/input/competitions/sebelas-maret-statistics-data-science-2026
Output: /kaggle/working
Mode  : validate


## 12. Jalankan mode yang dipilih

In [24]:
if MODE in {"validate", "fit", "all"} and not CONFIRM_TRAIN:
    raise RuntimeError(
        "Training diblokir. Ubah CONFIRM_TRAIN=True hanya setelah siap menjalankan training."
    )

result = run_pipeline(
    mode=MODE,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    save_feature_matrices=SAVE_FEATURE_MATRICES,
)
result

2026-07-16 11:59:28,623 | INFO | __main__ | Audit selesai: {'train_rows': 84396, 'test_rows': 21780, 'environment_rows': 888480, 'station_count': 30, 'train_start': '2023-01-01 06:00:00', 'train_end': '2025-09-18 18:00:00', 'test_start': '2025-09-19 06:00:00', 'test_end': '2026-05-18 18:00:00', 'environment_grid_complete': True, 'environment_missing_by_column': {'soil_moisture_0_7cm': 720, 'soil_moisture_7_28cm': 720, 'soil_moisture_28_100cm': 720, 'soil_moisture_100_255cm': 720, 'surface_pressure_hpa': 720, 'pressure_msl_hpa': 720, 'rmm1': 720, 'rmm2': 720, 'mjo_phase': 720, 'mjo_amplitude': 720, 'mjo_active': 720, 'nino_34': 12960}, 'solar_minus_999': 100080, 'rainfall_sources_identical': True}
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [12:00:17] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running o

{'mode': 'validate',
 'train_rows': 84396,
 'test_rows': 21780,
 'feature_count': 99,
 'validation': {'robust_weight': 0.7,
  'recommended_rounds': {'robust': 2870, 'squared': 521},
  'metrics': [{'fold': 'sep_2023',
    'rows': 20696,
    'anomalies': 7,
    'macro_station_mae': 1.0220120321497654,
    'worst_station_mae': 4.101457607646496,
    'raw_rmse': 2.7634321677877463,
    'raw_mae': 1.0292938797190303,
    'raw_bias': 0.7653182366599047,
    'raw_p95_absolute_error': 3.0895162383972448,
    'clean_rmse': 1.5510059430785328,
    'clean_mae': 0.9927902703367977,
    'clean_bias': 0.79080772084731,
    'clean_p95_absolute_error': 3.0858882464426656,
    'baseline_rmse': 2.792394105318451,
    'baseline_mae': 1.0774098549731133,
    'baseline_bias': 0.7528162635797376,
    'baseline_p95_absolute_error': 3.2624827220702883},
   {'fold': 'may_2024',
    'rows': 21313,
    'anomalies': 0,
    'macro_station_mae': 0.55386993086087,
    'worst_station_mae': 2.1810720061516156,
    'ra